# Paper figures — RRS / behavioral robustness

Run on Bluefin. No GPU needed; a login shell is enough.

```bash
conda activate slm && unset PYTHONPATH
cd /work/projects/phd-ssanjarip42/ssanjarip42/reasoning-robustness
jupyter lab --no-browser --port 8888
```

Outputs land in `figures/` as vector PDF (for LaTeX) plus PNG (for quick
viewing in the notebook).

In [6]:
pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 43.7 MB/s  0:00:00m0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.6 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lab 8.5 requires simplejson, which is not installed.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /work/projects/phd-ssanjarip42/ssanjarip42/jupyter_env/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use u

In [1]:
import json, glob, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

PROJ = Path("/work/projects/phd-ssanjarip42/ssanjarip42/reasoning-robustness")
FIGDIR = PROJ / "figures"
FIGDIR.mkdir(exist_ok=True)

print(matplotlib.__version__, pd.__version__)
print("figures ->", FIGDIR)

3.10.9 2.3.3
figures -> /work/projects/phd-ssanjarip42/ssanjarip42/reasoning-robustness/figures


## 1. Figure style

Colour convention, applied paper-wide:

| family | meaning | metrics |
|---|---|---|
| `Blues` | higher is better | accuracy, agreement, rationale provision, similarity |
| `Reds` | higher is worse | ASR, benign-flip rate, accuracy degradation |

A dark cell then never needs a caption to interpret.

In [2]:
POSITIVE, NEGATIVE = "Blues", "Reds"
FONT_SIZE = 8

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Times"],
    "font.size": FONT_SIZE,
    "axes.linewidth": 0.6,
    "pdf.fonttype": 42,   # IEEE rejects Type 3 fonts
    "ps.fonttype": 42,
})


def draw_heatmap(data, row_labels, col_labels, ax, cmap=POSITIVE,
                 vmin=None, vmax=None, cbar_label=None, fmt="{:.1f}",
                 sep_after=None):
    """Annotated heatmap. sep_after = column index to draw a rule after."""
    data = np.asarray(data, dtype=float)
    vmin = np.nanmin(data) if vmin is None else vmin
    vmax = np.nanmax(data) if vmax is None else vmax
    im = ax.imshow(data, cmap=plt.get_cmap(cmap),
                   norm=Normalize(vmin=vmin, vmax=vmax), aspect="auto")

    ax.set_xticks(np.arange(data.shape[1]))
    ax.set_yticks(np.arange(data.shape[0]))
    ax.set_xticklabels(col_labels, fontsize=FONT_SIZE)
    ax.set_yticklabels(row_labels, fontsize=FONT_SIZE)
    ax.tick_params(top=True, bottom=False, labeltop=True,
                   labelbottom=False, length=0)

    ax.set_xticks(np.arange(data.shape[1] + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0] + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1.4)
    ax.tick_params(which="minor", length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)

    thresh = vmin + 0.60 * (vmax - vmin)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            ax.text(j, i, fmt.format(v), ha="center", va="center",
                    fontsize=FONT_SIZE,
                    color="white" if v > thresh else "black")

    if sep_after is not None:
        ax.axvline(sep_after + 0.5, color="0.25", lw=1.0)

    cb = ax.figure.colorbar(im, ax=ax, fraction=0.030, pad=0.025)
    cb.outline.set_linewidth(0.5)
    cb.ax.tick_params(labelsize=FONT_SIZE - 1, width=0.5)
    if cbar_label:
        cb.set_label(cbar_label, fontsize=FONT_SIZE)
    return im


def save(fig, stem):
    fig.tight_layout(pad=0.3)
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"{stem}.{ext}", dpi=400, bbox_inches="tight")
    print("saved", stem)


MODELS = ["Gemma-4-E4B-it", "Qwen3.5-4B", "Qwen3.5-9B",
          "Llama-3.1-8B", "Llama-3.2-3B", "Phi-4-mini"]
ATTACKS = ["character", "word", "sentence", "multi"]
ATTACK_LABELS = ["Character", "Word", "Sentence", "Multi-level"]

## 2. Load matched clean--perturbed records

**Adjust `RECORDS_GLOB` and `COLS` to match your schema.** The next cell
prints the columns it actually found so you can check before aggregating.

Expected one row per (source message, model, prompt config, attack type,
seed), with the clean prediction joined on.

In [3]:
RECORDS_GLOB = str(PROJ / "outputs" / "**" / "*matched*.jsonl")

# map your column names -> the names used below
COLS = dict(
    msg_id      = "message_id",
    model       = "model",
    prompt      = "prompt_config",
    attack      = "attack_type",
    true_label  = "true_label",       # "smishing" / "ham"
    clean_pred  = "clean_pred",
    pert_pred   = "perturbed_pred",
)

paths = sorted(glob.glob(RECORDS_GLOB, recursive=True))
print(f"{len(paths)} file(s)")
for p in paths[:10]:
    print(" ", p)

df = pd.concat([pd.read_json(p, lines=True) for p in paths],
               ignore_index=True)
print("\nshape:", df.shape)
print("columns:", list(df.columns))
df.head(3)

3 file(s)
  /work/projects/phd-ssanjarip42/ssanjarip42/reasoning-robustness/outputs/analysis/dim2_matched/matched_pairs.jsonl
  /work/projects/phd-ssanjarip42/ssanjarip42/reasoning-robustness/outputs/analysis/dim2_matched_rationale_anywhere/matched_pairs_rationale_anywhere.jsonl
  /work/projects/phd-ssanjarip42/ssanjarip42/reasoning-robustness/outputs/analysis/dim2_rationale_test/matched_pairs_rationale_anywhere.jsonl

shape: (490004, 11)
columns: ['model', 'template', 'attack', 'seed', 'anchor_id', 'true_partner_id', 'baseline_partner_id', 'pred_label', 'true', 'base', 'gap']


,model,template,attack,seed,anchor_id,true_partner_id,baseline_partner_id,pred_label,true,base,gap
0,Qwen/Qwen3.5-4B,t3_neutral_explanation,character,0.0,msg-00001,msg-00001,msg-01468,smishing,"{'bertscore_f1': 0.920956, 'sentcos': 0.911013...","{'bertscore_f1': 0.8811669999999999, 'sentcos'...","{'bertscore_f1': 0.039789, 'sentcos': 0.303407..."
1,Qwen/Qwen3.5-4B,t3_neutral_explanation,character,1.0,msg-00001,msg-00001,msg-01627,smishing,"{'bertscore_f1': 0.9080079999999999, 'sentcos'...","{'bertscore_f1': 0.881369, 'sentcos': 0.771598...","{'bertscore_f1': 0.02664, 'sentcos': 0.102904,..."
2,Qwen/Qwen3.5-4B,t3_neutral_explanation,character,2.0,msg-00001,msg-00001,msg-00155,smishing,"{'bertscore_f1': 0.914032, 'sentcos': 0.846290...","{'bertscore_f1': 0.877867, 'sentcos': 0.569560...","{'bertscore_f1': 0.036164999999999996, 'sentco..."


In [12]:
print(COLS)
print(df.columns.tolist())

{'msg_id': 'message_id', 'model': 'model', 'prompt': 'prompt_config', 'attack': 'attack_type', 'true_label': 'true_label', 'clean_pred': 'clean_pred', 'pert_pred': 'perturbed_pred'}
['model', 'template', 'attack', 'seed', 'anchor_id', 'true_partner_id', 'baseline_partner_id', 'pred_label', 'true', 'base', 'gap']


In [11]:
# Remove accidental whitespace from imported column names
df.columns = df.columns.astype(str).str.strip()

rename_map = {v: k for k, v in COLS.items()}
df = df.rename(columns=rename_map)

required_cols = ["true_label", "clean_pred", "pert_pred"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(
        f"Missing required columns after renaming: {missing_cols}\n"
        f"Available columns: {df.columns.tolist()}\n"
        f"Rename mapping used: {rename_map}"
    )

for col in required_cols:
    df[col] = df[col].astype("string").str.strip().str.lower()

ValueError: Missing required columns after renaming: ['true_label', 'clean_pred', 'pert_pred']
Available columns: ['model', 'template', 'attack', 'seed', 'anchor_id', 'true_partner_id', 'baseline_partner_id', 'pred_label', 'true', 'base', 'gap']
Rename mapping used: {'message_id': 'msg_id', 'model': 'model', 'prompt_config': 'prompt', 'attack_type': 'attack', 'true_label': 'true_label', 'clean_pred': 'clean_pred', 'perturbed_pred': 'pert_pred'}

In [5]:
df = df.rename(columns={v: k for k, v in COLS.items()})

for col in ("true_label", "clean_pred", "pert_pred"):
    df[col] = df[col].astype(str).str.strip().str.lower()

print(df["true_label"].value_counts(), "\n")
print(df["attack"].value_counts(), "\n")
print(df["model"].value_counts())

KeyError: 'true_label'

## 3. Source-message-level rates

Both metrics are conditioned on the clean prediction being correct, then
averaged within each source message before averaging across messages, so
that a message contributing three seeds does not outweigh one contributing
a single deterministic paraphrase.

- **ASR** — over smishing messages: clean `smishing` -> perturbed `ham`
- **Benign-flip** — over ham messages: clean `ham` -> perturbed `smishing`

`by_prompt=True` reproduces Tables VI and VII; `by_prompt=False` pools
prompt configurations, which is the shape for a models x attack figure and
matches how Table VIII was built.

In [6]:
def flip_rate(df, true_label, from_label, to_label, by_prompt=False):
    """Message-weighted transition rate, conditioned on a correct clean pred."""
    sub = df[(df.true_label == true_label) & (df.clean_pred == from_label)].copy()
    sub["flipped"] = (sub.pert_pred == to_label).astype(float)

    keys = ["model", "attack"] + (["prompt"] if by_prompt else [])
    per_msg = sub.groupby(keys + ["msg_id"])["flipped"].mean()
    return per_msg.groupby(keys).mean().mul(100)


asr        = flip_rate(df, "smishing", "smishing", "ham")
benignflip = flip_rate(df, "ham",      "ham",      "smishing")

asr_wide = (asr.unstack("attack")
               .reindex(index=MODELS, columns=ATTACKS))
bf_wide  = (benignflip.unstack("attack")
                      .reindex(index=MODELS, columns=ATTACKS))

print("ASR (%)\n", asr_wide.round(2), "\n")
print("Benign-flip (%)\n", bf_wide.round(1))

AttributeError: 'DataFrame' object has no attribute 'true_label'

### Bootstrap confidence intervals (optional)

Matches the paper's stated procedure: 1000 class-stratified resamples at
the source-message level, all records for a sampled message resampled
together. Slow — run only for values you report numerically in text.

In [4]:
def bootstrap_ci(df, true_label, from_label, to_label,
                 model, attack, n_boot=1000, seed=0):
    sub = df[(df.true_label == true_label) &
             (df.clean_pred == from_label) &
             (df.model == model) & (df.attack == attack)].copy()
    sub["flipped"] = (sub.pert_pred == to_label).astype(float)
    per_msg = sub.groupby("msg_id")["flipped"].mean().values

    rng = np.random.default_rng(seed)
    if len(per_msg) == 0:
        return (np.nan, np.nan)
    draws = rng.choice(per_msg, size=(n_boot, len(per_msg)), replace=True)
    means = draws.mean(axis=1) * 100
    return tuple(np.percentile(means, [2.5, 97.5]).round(2))


for m in MODELS:
    lo, hi = bootstrap_ci(df, "smishing", "smishing", "ham", m, "character")
    print(f"{m:<18} character ASR 95% CI [{lo}, {hi}]")

NameError: name 'MODELS' is not defined

## 4. Figure — attack success rate

In [ ]:
fig, ax = plt.subplots(figsize=(3.45, 2.35))
draw_heatmap(asr_wide.values, MODELS, ATTACK_LABELS, ax,
             cmap=NEGATIVE, cbar_label="ASR (%)", fmt="{:.2f}")
save(fig, "fig_asr")

## 5. Figure — benign-flip rate

In [ ]:
fig, ax = plt.subplots(figsize=(3.45, 2.35))
draw_heatmap(bf_wide.values, MODELS, ATTACK_LABELS, ax,
             cmap=NEGATIVE, cbar_label="Benign-flip rate (%)")
save(fig, "fig_benign_flip")

## 6. Figure — clean-message accuracy

Typed from Table III rather than recomputed; swap in a groupby over the
clean records if you would rather derive it.

In [ ]:
acc = np.array([
    [95.6, 95.1, 94.9, 96.3, 96.1],
    [93.8, 94.9, 93.7, 94.9, 91.6],
    [90.1, 93.1, 89.6, 90.4, 87.2],
    [89.4, 89.0, 85.8, 93.7, 89.1],
    [89.3, 90.8, 91.3, 88.0, 87.2],
    [70.2, 70.0, 63.6, 74.5, 72.9],
])

fig, ax = plt.subplots(figsize=(3.45, 2.35))
draw_heatmap(acc, MODELS, ["Overall", "T1", "T2", "T3", "T4"], ax,
             cmap=POSITIVE, cbar_label="Accuracy (%)", sep_after=0)
save(fig, "fig_clean_accuracy")

## 7. Export the computed values

Keeps figure numbers and any numbers quoted in the text from drifting
apart.

In [ ]:
out = PROJ / "outputs" / "analysis" / "figure_values.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({
    "asr_by_model_attack": asr_wide.round(4).to_dict(),
    "benign_flip_by_model_attack": bf_wide.round(4).to_dict(),
}, indent=2))
print("wrote", out)

asr_wide.round(2).to_csv(PROJ / "outputs" / "analysis" / "asr_by_attack.csv")
bf_wide.round(2).to_csv(PROJ / "outputs" / "analysis" / "benign_flip_by_attack.csv")